# 06 · Scoring + MLflow + Contrato 2
Datos siempre en Spark — nunca toPandas sobre millones de filas.

In [0]:
%pip install lightgbm "numpy==1.26.4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 102.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Not uninstalling numpy at /databricks/python3/lib/python3.11/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-6b0fa857-3dab-48b2-af00-d1e8f2974333
    Can't uninstall 'numpy'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import numpy as np, pandas as pd, joblib
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
import mlflow, mlflow.sklearn
from mlflow.models import infer_signature

GOLD_PATH    = "/Volumes/workspace/default/e_commerce/gold_snapshot"
MODEL_PATH   = "/Volumes/workspace/default/e_commerce/propension_model.joblib"
TABLA_SALIDA = "workspace.default.contrato2"
RANDOM_STATE = 42; K_FINAL = 5; SAMPLE_KM = 300_000

BEST_PARAMS = dict(n_estimators=684, learning_rate=0.0197, num_leaves=211,
    max_depth=9, min_child_samples=197, subsample=0.9247,
    colsample_bytree=0.8086, reg_lambda=0.8359)

CLUSTER_FEATURES = ["total_views","distinct_products_viewed","brands_compared",
    "categories_explored","categories_explored_cid","browsing_duration_sec",
    "avg_price_viewed","max_price_viewed","electronics_view_share",
    "revisit_intensity","views_per_minute","avg_inter_event_sec"]
MODEL_FEATURES = CLUSTER_FEATURES + ["day_of_week","is_weekend",
    "sin_navegacion_previa","hour_sin","hour_cos"]
print("Config OK ✓")


/databricks/python/lib/python3.11/site-packages/mlflow/protos/service_pb2.py:11: UserWarning: google.protobuf.service module is deprecated. RPC implementations should provide code generator plugins which generate code specific to the RPC implementation. service.py will be removed in Jan 2025
  from google.protobuf import service as _service


Config OK ✓


## §1 MLflow setup

In [0]:
mlflow.set_tracking_uri("databricks")
try: mlflow.set_registry_uri("databricks-uc")
except: pass
mlflow.set_experiment("/Users/smartiner4@eafit.edu.co/propension_g8")
print("MLflow OK ✓")


MLflow OK ✓


## §2 Cargar Gold con Spark (lazy — sin toPandas masivo)

In [0]:
gold_spark = (spark.read.parquet(GOLD_PATH)
    .filter("label_window_corrupt = 0")
    .filter("session_date != '2019-11-14'")
    .withColumn("hour_sin", F.sin(2*F.lit(float(np.pi))*F.col("session_hour")/24))
    .withColumn("hour_cos", F.cos(2*F.lit(float(np.pi))*F.col("session_hour")/24)))

n = gold_spark.count()
print(f"Sesiones limpias: {n:,}")


Sesiones limpias: 19,711,743


## §3 Cargar modelo desde Volume

In [0]:
import sys, numpy, joblib

# El modelo se guardó con numpy 2.x (usa 'numpy._core'); en numpy 1.x es 'numpy.core'.
# Creamos alias para que joblib resuelva las referencias al cargar.
sys.modules["numpy._core"] = numpy.core
for sub in ["multiarray","umath","_multiarray_umath","numeric","numerictypes","overrides","fromnumeric"]:
    try:
        sys.modules[f"numpy._core.{sub}"] = __import__(f"numpy.core.{sub}", fromlist=[""])
    except Exception:
        pass

modelo = joblib.load(MODEL_PATH)
print("Modelo cargado ✓")


/databricks/python/lib/python3.11/site-packages/sklearn/base.py:347: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/databricks/python/lib/python3.11/site-packages/sklearn/base.py:347: InconsistentVersionWarning: Trying to unpickle estimator IsotonicRegression from version 1.6.1 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Modelo cargado ✓


/databricks/python/lib/python3.11/site-packages/sklearn/base.py:347: InconsistentVersionWarning: Trying to unpickle estimator CalibratedClassifierCV from version 1.6.1 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## §4 MLflow — registrar params y métricas

In [0]:
sample_pd = gold_spark.select(*MODEL_FEATURES).limit(5).toPandas().astype("float64")
sig = infer_signature(sample_pd, modelo.predict_proba(sample_pd)[:, 1])

with mlflow.start_run(run_name="lgbm_propension_final_g8"):
    mlflow.log_params(BEST_PARAMS)
    mlflow.log_param("calibracion",  "isotonic_cv3")
    mlflow.log_param("cuarentena",   "14-17 nov")
    mlflow.log_metric("cv_pr_auc_optuna", 0.1235)
    mlflow.log_metric("test_pr_auc",      0.1172)
    mlflow.log_metric("test_pr_auc_exBF", 0.1115)
    mlflow.log_metric("test_brier",       0.0515)
    mlflow.sklearn.log_model(modelo, "propension_model",
                             signature=sig, input_example=sample_pd)
    run_id = mlflow.active_run().info.run_id
print(f"Run: {run_id} ✓")


/databricks/python/lib/python3.11/site-packages/_distutils_hack/__init__.py:31: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


Run: 81a725c0bf9f49a49c8dbe4ba63806d3 ✓


In [ ]:
# §4.bis — VALIDACIÓN del modelo cargado: ¿reproduce los números del notebook 02?
# Chequeo barato sobre el TEST out-of-time (24-30 nov). Si NO reproduce ~0.1172 / ~0.0515,
# o si la prob media << tasa base, el problema es el desfase de versiones sklearn/numpy
# al des-serializar el joblib (ver warnings de §3) -> pinear versiones o re-exportar el modelo.
from sklearn.metrics import average_precision_score, brier_score_loss

val_pd = (gold_spark
          .filter("session_date > '2019-11-23'")          # test out-of-time
          .select(*MODEL_FEATURES, "target_purchase")
          .toPandas())
y_val = val_pd.pop("target_purchase").astype("int8")
p_val = modelo.predict_proba(val_pd[MODEL_FEATURES].astype("float64"))[:, 1]

print(f"Test n={len(y_val):,} | tasa base={y_val.mean():.4f}")
print(f"PR-AUC     = {average_precision_score(y_val, p_val):.4f}   (esperado ~0.1172)")
print(f"Brier      = {brier_score_loss(y_val, p_val):.4f}   (esperado ~0.0515)")
print(f"prob media = {p_val.mean():.4f}   (DEBE ≈ tasa base ~0.056; si es <<, calibración corrupta)")

## §5 Scoring via pandas UDF (sin sparkContext, sin broadcast)
El modelo se carga desde el Volume dentro del UDF — cada worker lo carga una vez por partición.

In [0]:
_MODEL_PATH     = MODEL_PATH
_MODEL_FEATURES = list(MODEL_FEATURES)

@F.pandas_udf(DoubleType())
def predict_udf(*cols):
    import sys, numpy, joblib, pandas as pd
    sys.modules["numpy._core"] = numpy.core
    for sub in ["multiarray","umath","_multiarray_umath","numeric","numerictypes","overrides","fromnumeric"]:
        try: sys.modules[f"numpy._core.{sub}"] = __import__(f"numpy.core.{sub}", fromlist=[""])
        except Exception: pass
    mdl = joblib.load(_MODEL_PATH)
    X = pd.concat(list(cols), axis=1).astype("float64")
    X.columns = _MODEL_FEATURES
    return pd.Series(mdl.predict_proba(X.values)[:, 1].astype("float64"))

scored_spark = gold_spark.withColumn(
    "prob_calibrada",
    predict_udf(*[F.col(f) for f in MODEL_FEATURES])
)
print("UDF de scoring definida ✓")


UDF de scoring definida ✓


## §6 KMeans — muestra pequeña en pandas + UDF para asignación

In [0]:
# Muestra de 300k para ajustar scaler + KMeans (cabe en memoria del driver)
samp_pd = (gold_spark.select(*CLUSTER_FEATURES)
           .sample(fraction=0.016, seed=RANDOM_STATE)
           .limit(SAMPLE_KM).toPandas().astype("float64"))
print(f"Muestra KMeans: {len(samp_pd):,} filas")

scaler = StandardScaler().fit(samp_pd)
km     = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10).fit(
             scaler.transform(samp_pd))

# Perfil y mapa número → nombre
perf = pd.DataFrame(scaler.inverse_transform(km.cluster_centers_), columns=CLUSTER_FEATURES)
def nombre(r):
    if r.electronics_view_share > 0.90: return "electronica_gama_media"
    if r.electronics_view_share > 0.60: return "electronica_premium"
    if r.total_views > 10:              return "explorador"
    if r.electronics_view_share < 0.05: return "general_bajo_valor"
    return "anonimo"
mapa = perf.apply(nombre, axis=1).to_dict()
print("Segmentos:", mapa)

# UDFs de segmento — objetos pequeños, van en el closure sin problema
_scaler          = scaler
_km              = km
_mapa            = dict(mapa)
_CLUSTER_FEATURES = list(CLUSTER_FEATURES)

@F.pandas_udf(IntegerType())
def segmento_udf(*cols):
    import pandas as pd
    X = pd.concat(list(cols), axis=1).astype("float64")
    X.columns = _CLUSTER_FEATURES
    return pd.Series(_km.predict(_scaler.transform(X)).astype("int32"))

@F.pandas_udf("string")
def nombre_udf(seg: pd.Series) -> pd.Series:
    return seg.map(_mapa)

final_spark = (scored_spark
    .withColumn("segmento",        segmento_udf(*[F.col(f) for f in CLUSTER_FEATURES]))
    .withColumn("segmento_nombre", nombre_udf(F.col("segmento"))))
print("UDFs de segmento definidas ✓")


Muestra KMeans: 300,000 filas


Exception ignored on calling ctypes callback function: <function _ThreadpoolInfo._find_modules_with_dl_iterate_phdr.<locals>.match_module_callback at 0xfff7100db920>
Traceback (most recent call last):
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 400, in match_module_callback
    self._make_module_from_path(filepath)
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 515, in _make_module_from_path
    module = module_class(filepath, prefix, user_api, internal_api)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 606, in __init__
    self.version = self.get_version()
                   ^^^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 646, in get_version
    config = get_config().split()
             ^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'split'
Ex

Segmentos: {0: 'electronica_gama_media', 1: 'explorador', 2: 'general_bajo_valor', 3: 'anonimo', 4: 'electronica_premium'}
UDFs de segmento definidas ✓


## §7 Contrato 2 → Delta  (aquí se ejecuta todo el pipeline Spark)

In [0]:
(final_spark
    .select("user_session","prob_calibrada","segmento","segmento_nombre")
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(TABLA_SALIDA))
print(f"\n✅ Contrato 2 persistido en: {TABLA_SALIDA}")



✅ Contrato 2 persistido en: workspace.default.contrato2


## §8 Verificación

In [0]:
ver = spark.table(TABLA_SALIDA)
print(f"Filas: {ver.count():,}")
ver.groupBy("segmento_nombre").count().orderBy("count", ascending=False).show()
ver.selectExpr("min(prob_calibrada)","max(prob_calibrada)","avg(prob_calibrada)").show()
print("\n✅ Listo para Kelly y Yeison.")


Filas: 19,711,743
+--------------------+-------+
|     segmento_nombre|  count|
+--------------------+-------+
|  general_bajo_valor|9827288|
|electronica_gama_...|5810725|
| electronica_premium|2687943|
|          explorador|1375291|
|             anonimo|  10496|
+--------------------+-------+

+--------------------+-------------------+--------------------+
| min(prob_calibrada)|max(prob_calibrada)| avg(prob_calibrada)|
+--------------------+-------------------+--------------------+
|6.686380686057824E-4| 0.5521581417732385|0.027481046543872976|
+--------------------+-------------------+--------------------+


✅ Listo para Kelly y Yeison.
